In [20]:
import os
import json
import requests
from PIL import Image, ExifTags
import time
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder
import pandas as pd


In [8]:
# CONSTS
HEADERS = {
    "User-Agent": (
        "ImageRecoProjectBot/1.0 "
        "(https://exemple-projet-univ.fr; contact: prenom.nom@etu-votre-ecole.fr) "
        "requests/2.31.0"
    ),
    "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
    "Accept-Language": "fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7",
}
BASE_DIR = "."
IMAGES_DIR = os.path.join(BASE_DIR, "images")
DATA_DIR = os.path.join(BASE_DIR, "data")
METADATA_PATH = os.path.join(DATA_DIR, "images_metadata.json")
LABELS_PATH = os.path.join(DATA_DIR, "images_labels.json")

# Création des dossiers data et images
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

In [8]:
url_api = "https://commons.wikimedia.org/w/api.php?action=query&generator=search&gsrsearch=cat&gsrnamespace=6&gsrlimit=50&prop=imageinfo&iiprop=url|size|mime|extmetadata&format=json"

def search_commons_images(query, headers, limit=50):
    """
    Cherche des images sur Wikimedia Commons et renvoie une liste
    de dicts avec url + licence + auteur, etc.
    """
    params = {
        "action": "query",
        "generator": "search",
        "gsrsearch": query,
        "gsrnamespace": 6,        # namespace 'File:'
        "gsrlimit": limit,
        "prop": "imageinfo",
        "iiprop": "url|size|mime|extmetadata",
        "format": "json",
        "origin": "*"
    }

    resp = requests.get(url_api, params=params, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    if "query" not in data or "pages" not in data["query"]:
        return []

    images = []
    for page in data["query"]["pages"].values():
        title = page.get("title")  # "File:Something.jpg"
        imageinfo = page.get("imageinfo", [])
        if not imageinfo:
            continue
        info = imageinfo[0]
        url = info.get("url")
        width = info.get("width")
        height = info.get("height")
        mime = info.get("mime")
        extmeta = info.get("extmetadata", {})

        # Métadonnées de licence / auteur (si disponibles)
        license_short_name = extmeta.get("LicenseShortName", {}).get("value")
        license_url = extmeta.get("LicenseUrl", {}).get("value")
        artist = extmeta.get("Artist", {}).get("value")
        credit = extmeta.get("Credit", {}).get("value")

        images.append({
            "title": title,
            "url": url,
            "width": width,
            "height": height,
            "mime": mime,
            "license_short_name": license_short_name,
            "license_url": license_url,
            "artist": artist,
            "credit": credit,
            "source": "Wikimedia Commons"
        })

    return images

In [9]:
all_sources = []
topics = [
    ("cat", 40),
    ("dog", 40),
    ("mountain landscape", 40),
]

for query, limit in topics:
    results = search_commons_images(query, HEADERS, limit=limit)
    all_sources.extend(results)

print(f"{len(all_sources)} images trouvées avant dédoublonnage")

120 images trouvées avant dédoublonnage


In [13]:
def download_image(url, dest_folder, idx):
    response = requests.get(url, headers=HEADERS, timeout=10)
    response.raise_for_status()
    ext = os.path.splitext(url.split("?")[0])[1] or ".jpg"
    filename = f"img_{idx:03d}{ext}"
    filepath = os.path.join(dest_folder, filename)
    with open(filepath, "wb") as f:
        f.write(response.content)
    return filename, filepath

def extract_exif(image_path):
        try:
            img = Image.open(image_path)
            exif_data = img._getexif()
            if not exif_data:
                return {}
            exif = {}
            for tag_id, value in exif_data.items():
                tag = ExifTags.TAGS.get(tag_id, tag_id)
                exif[tag] = str(value)
            return exif
        except Exception:
            return {}

metadata_list = []

for idx, src in enumerate(all_sources, start=1):
    try:
        url = src["url"]
        license_info = src.get("license", "unknown")
        source_site = src.get("source", "unknown")

        filename, filepath = download_image(url, IMAGES_DIR, idx)
        time.sleep(0.5)
        with Image.open(filepath) as img:
            width, height = img.size
            img_format = img.format

        file_size_bytes = os.path.getsize(filepath)
        file_size_kb = round(file_size_bytes / 1024, 2)

        exif = extract_exif(filepath)

        metadata = {
            "filename": filename,
            "width": width,
            "height": height,
            "format": img_format,
            "file_size_kb": file_size_kb,
            "url": url,
            "license": license_info,
            "source": source_site,
            "exif": exif
        }
        metadata_list.append(metadata)

    except Exception as e:
        print(f"Erreur sur l’image {idx} ({src.get('url')}): {e}")

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata_list, f, ensure_ascii=False, indent=2)

print(f"{len(metadata_list)} images traitées et sauvegardées dans {METADATA_PATH}")


Erreur sur l’image 12 (https://upload.wikimedia.org/wikipedia/commons/a/ac/Cat_lapping_water_off_ground_in_slow_motion.gk.webm): cannot identify image file '.\\images\\img_012.webm'
Erreur sur l’image 60 (https://upload.wikimedia.org/wikipedia/commons/e/e6/Dog_penis.ogv): cannot identify image file '.\\images\\img_060.ogv'
Erreur sur l’image 73 (https://upload.wikimedia.org/wikipedia/commons/8/82/P_S_Kr%C3%B8yer_1899_-_Sommeraften_ved_Skagens_strand._Kunstneren_og_hans_hustru.jpg): Image size (191882294 pixels) exceeds limit of 178956970 pixels, could be decompression bomb DOS attack.


c:\Users\antoi\Documents\Cours\DMML\MachineLearning\env\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))


117 images traitées et sauvegardées dans .\data\images_metadata.json


In [5]:
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata_list = json.load(f)

def compute_orientation(width, height):
    if width > height:
        return "paysage"
    elif height > width:
        return "portrait"
    else:
        return "carré"

def compute_size_category(max_dim):
    if max_dim < 500:
        return "vignette"
    elif max_dim <= 1500:
        return "moyenne"
    else:
        return "grande"

In [ ]:
def get_dominant_colors(image_path, n_colors=4):
    img = Image.open(image_path).convert("RGB")
    img_small = img.resize((150, 150))  # pour aller plus vite
    pixels = np.array(img_small).reshape(-1, 3)

    kmeans = KMeans(n_clusters=n_colors, n_init=5)
    kmeans.fit(pixels)
    centers = kmeans.cluster_centers_.astype(int)

    # Retourne une liste de listes [[r,g,b], ...]
    return [center.tolist() for center in centers]

def rgb_to_basic_name(rgb):
    r, g, b = rgb
    # Version ultra simple à raffiner si vous voulez
    if max(rgb) < 80:
        return "noir"
    if r > g and r > b:
        return "rouge"
    if g > r and g > b:
        return "vert"
    if b > r and b > g:
        return "bleu"
    return "gris"

labels = {}

for meta in metadata_list:
    filename = meta["filename"]
    width = meta["width"]
    height = meta["height"]
    orientation = compute_orientation(width, height)
    size_cat = compute_size_category(max(width, height))

    image_path = os.path.join(IMAGES_DIR, filename)
    try:
        colors = get_dominant_colors(image_path, n_colors=4)
        color_names = [rgb_to_basic_name(c) for c in colors]
    except Exception as e:
        print(f"Impossible d'extraire les couleurs pour {filename}: {e}")
        colors = []
        color_names = []

    labels[filename] = {
        "predominant_colors": colors,
        "color_names": color_names,
        "orientation": orientation,
        "size_category": size_cat,
        "tags": []
    }

with open(LABELS_PATH, "w", encoding="utf-8") as f:
    json.dump(labels, f, ensure_ascii=False, indent=2)

print(f"Annotations sauvegardées dans {LABELS_PATH}")

Annotations sauvegardées dans .\data\images_labels.json


In [13]:
import json
import os
import random
from collections import Counter

DATA_DIR = "data"
LABELS_PATH = os.path.join(DATA_DIR, "images_labels.json")
USERS_PATH = os.path.join(DATA_DIR, "users.json")

# Charger les labels
with open(LABELS_PATH, "r", encoding="utf-8") as f:
    image_labels = json.load(f)

image_filenames = list(image_labels.keys())
print(f"{len(image_filenames)} images avec labels chargées.")

# Fonctions utilitaires

def most_common_or_none(lst):
    if not lst:
        return None
    return Counter(lst).most_common(1)[0][0]

def build_user_profile(user_id, favorite_images):
    colors = []
    orientations = []
    sizes = []
    tags = []

    for fname in favorite_images:
        info = image_labels[fname]
        colors.extend(info.get("color_names", []))
        orientations.append(info.get("orientation"))
        sizes.append(info.get("size_category"))
        tags.extend(info.get("tags", []))

    profile = {
        "user_id": user_id,
        "favorite_colors": list({c for c in colors if c}),  # ensemble ⇢ liste
        "favorite_orientation": most_common_or_none(
            [o for o in orientations if o]
        ),
        "favorite_size": most_common_or_none(
            [s for s in sizes if s]
        ),
        "favorite_tags": list({t for t in tags if t}),
        "favorite_images": favorite_images,
    }
    return profile

# Créer des listes d'images filtrées pour des goûts différents
# (à adapter selon ton dataset réel)

def filter_images_by_color(color):
    return [
        fname for fname, info in image_labels.items()
        if color in info.get("color_names", [])
    ]

def filter_images_by_orientation(orientation):
    return [
        fname for fname, info in image_labels.items()
        if info.get("orientation") == orientation
    ]

def filter_images_by_size(size_cat):
    return [
        fname for fname, info in image_labels.items()
        if info.get("size_category") == size_cat
    ]

# Créer 5 profils d'utilisateurs avec des préférences différentes
users = {}

random.seed(42)  # pour reproductibilité

# User 1 : aime les images "rouges" en paysage
candidates_u1 = list(
    set(filter_images_by_color("rouge"))
    & set(filter_images_by_orientation("paysage"))
)
favorites_u1 = random.sample(
    candidates_u1, k=min(12, len(candidates_u1))
) if candidates_u1 else random.sample(image_filenames, k=12)
users["user_001"] = build_user_profile("user_001", favorites_u1)

# User 2 : aime les images "bleues" en portrait
candidates_u2 = list(
    set(filter_images_by_color("bleu"))
    & set(filter_images_by_orientation("portrait"))
)
favorites_u2 = random.sample(
    candidates_u2, k=min(12, len(candidates_u2))
) if candidates_u2 else random.sample(image_filenames, k=12)
users["user_002"] = build_user_profile("user_002", favorites_u2)

# User 3 : aime les grandes images (size_category = grande)
candidates_u3 = filter_images_by_size("grande")
favorites_u3 = random.sample(
    candidates_u3, k=min(12, len(candidates_u3))
) if candidates_u3 else random.sample(image_filenames, k=12)
users["user_003"] = build_user_profile("user_003", favorites_u3)

# User 4 : aime les petites vignettes (size_category = vignette)
candidates_u4 = filter_images_by_size("vignette")
favorites_u4 = random.sample(
    candidates_u4, k=min(12, len(candidates_u4))
) if candidates_u4 else random.sample(image_filenames, k=12)
users["user_004"] = build_user_profile("user_004", favorites_u4)

# User 5 : goûts mixtes, tirés aléatoirement
favorites_u5 = random.sample(
    image_filenames, k=min(12, len(image_filenames))
)
users["user_005"] = build_user_profile("user_005", favorites_u5)

# Sauvegarde dans users.json
with open(USERS_PATH, "w", encoding="utf-8") as f:
    json.dump(users, f, ensure_ascii=False, indent=2)

print(f"{len(users)} profils utilisateurs sauvegardés dans {USERS_PATH}")

117 images avec labels chargées.
5 profils utilisateurs sauvegardés dans data\users.json


In [18]:
with open(LABELS_PATH, "r", encoding="utf-8") as f:
    image_labels = json.load(f)

rows = []
for fname, info in image_labels.items():
    color_names = info.get("color_names", [])
    main_color = color_names[0] if color_names else None

    tags = info.get("tags", [])
    main_tag = tags[0] if tags else None

    rows.append({
        "filename": fname,
        "main_color": main_color,
        "orientation": info.get("orientation"),
        "size_category": info.get("size_category"),
        "main_tag": main_tag,
    })

df_images = pd.DataFrame(rows)
print(df_images.head())
print(df_images["main_color"].value_counts())

      filename main_color orientation size_category main_tag
0  img_001.jpg      rouge       carré        grande     None
1  img_002.jpg      rouge    portrait        grande     None
2  img_003.jpg       bleu    portrait        grande     None
3  img_004.jpg      rouge     paysage        grande     None
4  img_005.jpg       noir     paysage        grande     None
main_color
rouge    49
noir     31
bleu     16
vert     16
gris      5
Name: count, dtype: int64


In [22]:
feature_cols = ["main_color", "orientation", "size_category", "main_tag"]

encoders = {}
encoded_features = []

for col in feature_cols:
    le = LabelEncoder()
    values = df_images[col].fillna("unknown")
    encoded_col = le.fit_transform(values)
    encoders[col] = le
    encoded_features.append(encoded_col)

X = np.vstack(encoded_features).T  # shape (n_images, n_features)

# Choix du nombre de clusters
k = 5
kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
df_images["cluster"] = kmeans.fit_predict(X)

df_images.head()

,filename,main_color,orientation,size_category,main_tag,cluster
0,img_001.jpg,rouge,carré,grande,None,0
1,img_002.jpg,rouge,portrait,grande,None,2
2,img_003.jpg,bleu,portrait,grande,None,1
3,img_004.jpg,rouge,paysage,grande,None,0
4,img_005.jpg,noir,paysage,grande,None,3


In [24]:
with open(LABELS_PATH, "r", encoding="utf-8") as f:
    image_labels = json.load(f)

with open(USERS_PATH, "r", encoding="utf-8") as f:
    users = json.load(f)

all_images = list(image_labels.keys())

def get_user_top_clusters(user_id, top_k=2):
    user = users[user_id]
    favs = user["favorite_images"]

    clusters = []
    for fname in favs:
        info = image_labels.get(fname)
        if info is None:
            continue
        cluster = info.get("cluster")
        if cluster is not None:
            clusters.append(cluster)

    if not clusters:
        return []

    counts = Counter(clusters).most_common(top_k)
    # renvoie seulement les id de cluster
    return [c for c, _ in counts]

In [25]:
def recommend_images(user_id, n_recommendations=5):
    user = users[user_id]
    favs = set(user["favorite_images"])
    top_clusters = get_user_top_clusters(user_id, top_k=2)

    # Si on n'a pas de clusters (cas extrême), recommander au hasard
    if not top_clusters:
        candidates = [img for img in all_images if img not in favs]
    else:
        candidates = []
        for fname, info in image_labels.items():
            if fname in favs:
                continue
            if info.get("cluster") in top_clusters:
                candidates.append(fname)

    # Petit bonus : essayer de respecter les couleurs préférées si dispo
    fav_colors = set(user.get("favorite_colors") or [])

    def score_image(fname):
        info = image_labels[fname]
        colors = set(info.get("color_names") or [])
        # score = nombre de couleurs en commun
        return len(colors & fav_colors)

    # Trier les candidats par score décroissant, puis mélanger un peu
    candidates_sorted = sorted(candidates, key=score_image, reverse=True)

    if len(candidates_sorted) > n_recommendations:
        candidates_sorted = candidates_sorted[: n_recommendations * 3]
        random.shuffle(candidates_sorted)

    selected = candidates_sorted[:n_recommendations]

    # Construire une petite explication pour chaque image
    recommendations = []
    for fname in selected:
        info = image_labels[fname]
        reason_parts = []

        # couleur
        colors = info.get("color_names") or []
        if fav_colors and set(colors) & fav_colors:
            reason_parts.append(
                f"couleurs proches de vos préférences ({', '.join(sorted(set(colors) & fav_colors))})"
            )

        # orientation / taille
        ori = info.get("orientation")
        size = info.get("size_category")
        if ori:
            reason_parts.append(f"orientation {ori}")
        if size:
            reason_parts.append(f"taille {size}")

        # cluster
        cl = info.get("cluster")
        if cl is not None:
            reason_parts.append(f"cluster {cl}")

        reason = " ; ".join(reason_parts) if reason_parts else "similaire à vos images favorites"
        recommendations.append((fname, reason))

    return recommendations

In [26]:
recs = recommend_images("user_001", 5)
for fname, reason in recs:
    print(fname, "→", reason)

img_019.JPG → couleurs proches de vos préférences (gris, noir, rouge) ; orientation portrait ; taille grande
img_023.jpg → couleurs proches de vos préférences (noir, rouge, vert) ; orientation paysage ; taille grande
img_006.jpg → couleurs proches de vos préférences (noir, rouge, vert) ; orientation portrait ; taille grande
img_110.jpg → couleurs proches de vos préférences (bleu, noir, rouge, vert) ; orientation paysage ; taille grande
img_007.jpg → couleurs proches de vos préférences (bleu, noir, rouge) ; orientation portrait ; taille grande
